# Bayesian Optimization History Plotting

This notebook visualizes the results from BO hyperparameter tuning runs.
It reads CSV history files saved by `bo_hst_nn.py` and creates summary plots.

In [ ]:
from pathlib import Path
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Load History Data

Specify the path to your saved history CSV file.

In [ ]:
# Update this path to your specific history file
# Example: history_path = Path("output_log/bo_hst_nn_history_0410_123456.csv")
history_path = Path("output_log/bo_hst_nn_history_MMDD_HHMMSS.csv")

# Or list all available history files
output_dir = Path("output_log")
if output_dir.exists():
    history_files = sorted(output_dir.glob("bo_hst_nn_history_*.csv"))
    print(f"Found {len(history_files)} history file(s):")
    for i, file in enumerate(history_files, 1):
        print(f"  {i}. {file.name}")
    
    if history_files:
        # Use the most recent file by default
        history_path = history_files[-1]
        print(f"\nUsing most recent: {history_path.name}")

In [ ]:
# Load the history data
history_df = pd.read_csv(history_path)

# Process the data
history_df["eval_index"] = np.arange(1, len(history_df) + 1)

# Parse hidden_sizes from string representation back to list
history_df["hidden_sizes"] = history_df["hidden_sizes"].apply(eval)
history_df["hidden_label"] = history_df["hidden_sizes"].apply(
    lambda values: "-".join(str(value) for value in values)
)
history_df["num_layers"] = history_df["hidden_sizes"].apply(len)
history_df["log10_learning_rate"] = np.log10(history_df["learning_rate"])
history_df["running_best_eval_mse"] = history_df["best_eval_mse"].cummin()
history_df["improved"] = (
    history_df["best_eval_mse"] == history_df["running_best_eval_mse"]
)

print(f"Loaded {len(history_df)} evaluations")
print(f"Best validation MSE: {history_df['best_eval_mse'].min():.6e}")
history_df.head()

## Plot 1: Validation MSE by Evaluation

Shows each evaluated configuration and the incumbent best over time.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(
    history_df["eval_index"],
    history_df["best_eval_mse"],
    color="tab:blue",
    alpha=0.75,
    label="Evaluated config",
)
ax.plot(
    history_df["eval_index"],
    history_df["running_best_eval_mse"],
    color="tab:red",
    linewidth=2,
    label="Best so far",
)
ax.set_xlabel("BO Evaluation")
ax.set_ylabel("Validation MSE")
ax.set_title("HST NN Hyperparameter BO")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()

# Optionally save
# plt.savefig("plots/bo_hst_nn_incumbent.png")
plt.show()

## Plot 2: Hyperparameter Trace

Shows how hyperparameters evolved during the BO process, with point color indicating `best_eval_mse`.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

scatter_kwargs = {
    "c": history_df["best_eval_mse"],
    "cmap": "viridis_r",
    "s": 70,
    "alpha": 0.85,
}

scatter = axes[0].scatter(
    history_df["eval_index"],
    history_df["log10_learning_rate"],
    **scatter_kwargs,
)
axes[0].set_ylabel("log10(LR)")
axes[0].grid(True, alpha=0.3)

axes[1].scatter(
    history_df["eval_index"],
    history_df["batch_size"],
    **scatter_kwargs,
)
axes[1].set_ylabel("Batch Size")
axes[1].grid(True, alpha=0.3)

architecture_codes, architecture_labels = pd.factorize(history_df["hidden_label"])
axes[2].scatter(
    history_df["eval_index"],
    architecture_codes,
    **scatter_kwargs,
)
axes[2].set_yticks(np.arange(len(architecture_labels)))
axes[2].set_yticklabels(architecture_labels)
axes[2].set_ylabel("Architecture")
axes[2].set_xlabel("BO Evaluation")
axes[2].grid(True, alpha=0.3)

cbar = plt.colorbar(scatter, ax=axes)
cbar.set_label("best_eval_mse")

# Mark improvements with vertical lines
improvement_indices = history_df.loc[history_df["improved"], "eval_index"]
for ax in axes:
    for eval_index in improvement_indices:
        ax.axvline(eval_index, color="0.8", linestyle="--", linewidth=0.8)

fig.suptitle("HST NN BO Hyperparameter Trace", y=0.98)
plt.tight_layout()

# Optionally save
# plt.savefig("plots/bo_hst_nn_trace.png")
plt.show()

## Plot 3: Hyperparameter Scatter

Learning rate vs batch size, colored by validation MSE. Point size indicates number of hidden layers.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
scatter = ax.scatter(
    history_df["log10_learning_rate"],
    history_df["batch_size"],
    c=history_df["best_eval_mse"],
    cmap="viridis_r",
    s=80 + 25 * (history_df["num_layers"] - 1),
    alpha=0.85,
)
for _, row in history_df.iterrows():
    ax.annotate(
        row["hidden_label"],
        (row["log10_learning_rate"], row["batch_size"]),
        textcoords="offset points",
        xytext=(4, 4),
        fontsize=8,
        alpha=0.8,
    )
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Validation MSE")
ax.set_xlabel("log10(Learning Rate)")
ax.set_ylabel("Batch Size")
ax.set_title("HST NN BO Performance by Hyperparameter")
ax.grid(True, alpha=0.3)
plt.tight_layout()

# Optionally save
# plt.savefig("plots/bo_hst_nn_scatter.png")
plt.show()

## Summary Statistics

In [ ]:
print("Best configuration:")
best_idx = history_df["best_eval_mse"].idxmin()
best_row = history_df.loc[best_idx]
print(f"  Hidden sizes: {best_row['hidden_sizes']}")
print(f"  Batch size: {best_row['batch_size']}")
print(f"  Learning rate: {best_row['learning_rate']:.2e}")
print(f"  Validation MSE: {best_row['best_eval_mse']:.6e}")
print(f"  Found at evaluation: {best_row['eval_index']:.0f}")

print("\nImprovement timeline:")
improvements = history_df[history_df["improved"]]
for _, row in improvements.iterrows():
    print(f"  Eval {row['eval_index']:.0f}: MSE={row['best_eval_mse']:.6e} ({row['hidden_label']}, bs={row['batch_size']:.0f}, lr={row['learning_rate']:.2e})")